# P5-3 · GĐ 3 · Điều khiển bằng chữ

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import math, torch, torch.nn as nn

def time_emb(t, dim=32):                              # như GĐ1
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half) / half)
    ang = t.float().unsqueeze(1) * freqs.unsqueeze(0)
    return torch.cat([ang.sin(), ang.cos()], dim=1)

class CondDenoiser(nn.Module):
    def __init__(self, n_cls=3, h=256):
        super().__init__()
        self.lab = nn.Embedding(n_cls + 1, 32)          # chỉ số n_cls là nhãn rỗng ∅
        self.net = nn.Sequential(nn.Linear(2 + 32, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, x, t, y):
        return self.net(torch.cat([x, time_emb(t) + self.lab(y)], 1))   # cộng nhãn vào thời gian

model = CondDenoiser()
x, t, y = torch.randn(5, 2), torch.randint(0, 1000, (5,)), torch.tensor([0, 1, 2, 0, 3])   # nhãn 3 = ∅
model(x, t, y).shape

torch.Size([5, 2])


In [ ]:
NULL_ID = 3
y = torch.randint(0, 3, (10_000,))                                         # nhãn thật của một batch lớn
y = torch.where(torch.rand(len(y)) < 0.1, torch.full_like(y, NULL_ID), y)   # 10% bỏ nhãn
(y == NULL_ID).float().mean().item()                                       # khoảng 0.1

0.09790000319480896


In [ ]:
x, t, w = torch.randn(4, 2), torch.full((4,), 500), 3.0
y, y_null = torch.zeros(4, dtype=torch.long), torch.full((4,), NULL_ID)
e_cond, e_uncond = model(x, t, y), model(x, t, y_null)
eps = e_uncond + w * (e_cond - e_uncond)
eps.shape                                  # cùng shape như một lần gọi model

torch.Size([4, 2])
